# Real Word2Vec with a Familiar Keras ANN
**A simple `Sequential` network with two `Dense` layers.** We train a classic Skip-Gram full-softmax model on real WikiText sentences. No Gensim, no custom model class, no manual gradient code.

**Network:** One-hot center word → `Dense(64, linear)` → `Dense(vocabulary_size, softmax)` → predict context word.

The first Dense layer's weights are the learned word embeddings. Full softmax is easy to teach but slower than negative sampling, so the vocabulary and training pairs are capped for Colab CPU.

In [ ]:
!pip -q install datasets matplotlib scikit-learn

## 1. Load real text and build a vocabulary
We use a subset of the actual WikiText-2 training corpus to keep the full-softmax ANN practical.

In [ ]:
import re
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from collections import Counter
from datasets import load_dataset
from sklearn.decomposition import PCA

VOCAB_SIZE = 2000     # Smaller vocabulary = faster softmax.
DIM = 64              # Embedding dimensions (hidden neurons).
WINDOW = 2            # Number of context positions on each side.
MAX_PAIRS = 40000     # Limit training examples for Colab CPU.
EPOCHS = 5

tf.random.set_seed(42)
raw = load_dataset('Salesforce/wikitext', 'wikitext-2-raw-v1', split='train')
sentences = [re.findall(r'[a-z]+', line.lower()) for line in raw['text']]
sentences = [s for s in sentences if len(s) >= 3]
counts = Counter(word for sentence in sentences for word in sentence)
vocab = [w for w, count in counts.most_common(VOCAB_SIZE) if count >= 3]
word_id = {w: i for i, w in enumerate(vocab)}
V = len(vocab)
print('Vocabulary size:', V)
print('Example sentence:', sentences[0][:15])

## 2. Create Skip-Gram training examples
For the sentence `the cat sleeps`, examples include `cat → the` and `cat → sleeps`. The input is the **center word**; the label is a nearby **context word**.

In [ ]:
pairs = []
for sentence in sentences:
    ids = [word_id[w] for w in sentence if w in word_id]
    for i, center in enumerate(ids):
        for j in range(max(0, i-WINDOW), min(len(ids), i+WINDOW+1)):
            if i != j:
                pairs.append((center, ids[j]))
        if len(pairs) >= MAX_PAIRS:
            break
    if len(pairs) >= MAX_PAIRS:
        break

pairs = np.array(pairs[:MAX_PAIRS], dtype='int32')
centers = pairs[:, 0]
contexts = pairs[:, 1]
print('Training pairs:', len(pairs))
print('Example:', vocab[centers[0]], '→', vocab[contexts[0]])

## 3. Build the ANN with `Sequential` and `Dense`

- **Input:** a one-hot vector of length `V`.
- **Hidden layer:** `Dense(DIM, activation='linear', use_bias=False)` produces the word embedding.
- **Output:** `Dense(V, activation='softmax', use_bias=False)` predicts a probability for every possible context word.

This is classic Skip-Gram with **full softmax** (not negative sampling). It is a shallow ANN with no hidden nonlinearity.

In [ ]:
model = tf.keras.Sequential([
    tf.keras.Input(shape=(V,)),
    tf.keras.layers.Dense(DIM, activation='linear', use_bias=False, name='Embedding'),
    tf.keras.layers.Dense(V, activation='softmax', use_bias=False, name='Predict_Context')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
model.summary()
print('Embedding matrix shape:', model.get_layer('Embedding').get_weights()[0].shape)

## 4. Train the ANN using `model.fit()`
`tf.one_hot()` converts word IDs to one-hot vectors **one batch at a time**, avoiding a large one-hot array in RAM. Keras performs forward propagation, computes the loss, and updates both Dense layers by backpropagation.

In [ ]:
# Convert center word IDs into one-hot vectors batch by batch.
train_data = tf.data.Dataset.from_tensor_slices((centers, contexts))
train_data = train_data.shuffle(10000, seed=42).batch(128)
train_data = train_data.map(lambda x, y: (tf.one_hot(x, V), y))

before = model.get_layer('Embedding').get_weights()[0].copy()
history = model.fit(train_data, epochs=EPOCHS)
after = model.get_layer('Embedding').get_weights()[0]

plt.plot(history.history['loss'], marker='o')
plt.xlabel('Epoch'); plt.ylabel('Cross-entropy loss'); plt.title('ANN training'); plt.grid(alpha=0.3); plt.show()
print('Example word:', vocab[0])
print('Before:', np.round(before[0, :5], 3))
print('After: ', np.round(after[0, :5], 3))

## 5. Inspect the trained word vectors
Each row of the first Dense layer's weight matrix is one learned word embedding. Compare cosine similarity and nearest neighbors.

In [ ]:
vectors = model.get_layer('Embedding').get_weights()[0]
normalized = vectors / np.maximum(np.linalg.norm(vectors, axis=1, keepdims=True), 1e-8)

def nearest(word, k=5):
    if word not in word_id:
        return 'Not in vocabulary'
    scores = normalized @ normalized[word_id[word]]
    order = np.argsort(-scores)
    return [(vocab[i], round(float(scores[i]), 3)) for i in order if vocab[i] != word][:k]

for word in ['king', 'computer', 'city', 'music']:
    print(word, nearest(word))

## 6. Try vector arithmetic
These analogies are experiments, not guaranteed results: a 2,000-word vocabulary and short training often do not learn reliable relationships.

In [ ]:
words = ['king', 'man', 'woman']
if all(w in word_id for w in words):
    query = vectors[word_id['king']] - vectors[word_id['man']] + vectors[word_id['woman']]
    scores = normalized @ (query / max(np.linalg.norm(query), 1e-8))
    print('king - man + woman:', [(vocab[i], round(float(scores[i]), 3)) for i in np.argsort(-scores) if vocab[i] not in words][:5])
else:
    print('Some analogy words are outside the top', V, 'words.')

## 7. Visualize embeddings with PCA
PCA projects high-dimensional vectors into 2D; the projection is approximate.

In [ ]:
words = [w for w in ['king','queen','man','woman','dog','cat','computer','science','music','city','country','france'] if w in word_id]
if len(words) >= 3:
    points = PCA(n_components=2).fit_transform(np.array([vectors[word_id[w]] for w in words]))
    plt.figure(figsize=(9, 6))
    plt.scatter(points[:, 0], points[:, 1])
    for word, (x, y) in zip(words, points):
        plt.annotate(word, (x, y), xytext=(5, 5), textcoords='offset points')
    plt.title('Word embeddings learned by the ANN'); plt.grid(alpha=0.3); plt.show()

## Student challenges
1. Change `DIM=64` to `DIM=16` or `DIM=100`.
2. Increase `MAX_PAIRS` and `EPOCHS` and compare neighbors.
3. Change `WINDOW=2` to `WINDOW=5`.
4. Why is the first Dense layer's weight matrix the embedding table?
5. Why does full softmax become expensive when `VOCAB_SIZE` grows?

**Teacher note:** This notebook prioritizes a familiar ANN over training speed. For larger corpora, negative sampling or hierarchical softmax is preferable.